In [1]:
# Integrating Machine Learning Models into APIs

In [4]:
from fastapi import FastAPI
import uvicorn
import threading
import time
import requests

app = FastAPI(
    title = "My First Fast API",
    description = "This is my first Fast API",
    version = "0.0.1"
)

In [7]:
PORT = 8001

def run_server():
  uvicorn.run(
      app,
      host = "127.0.0.1",
      port = PORT,
      log_level = "warning"
  )
server_thread = threading.Thread(target = run_server,daemon = True)
server_thread.start()
time.sleep(1)

print("FastAPI SErver Started")
URL = "http://127.0.0.1:8001"

FastAPI SErver Started


In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Load the Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [5]:
from pydantic import BaseModel, Field

class IrisInput(BaseModel):
  sepal_length : float = Field(gt=0)
  sepal_width : float = Field(gt=0)
  petal_length : float = Field(gt=0)
  petal_width : float= Field(gt=0)

In [6]:
@app.post("/predict_iris")

def predict_iris(iris_input : IrisInput):
  input_data = [[
      iris_input.sepal_length,
      iris_input.sepal_width,
      iris_input.petal_length,
      iris_input.petal_width
  ]]

  prediction = classifier.predict(input_data)

  probailities = classifier.predict_proba(input_data)

  return {
      "prediction" : int(prediction[0]),
      "probabilities" : probailities.tolist()[0]
  }


In [7]:
sample = {
          "sepal_length" : 5.1,
          "sepal_width" : 3.5,
          "petal_length" : 1.4,
          "petal_width" : 0.2
}

reponse = requests.post(
    URL+"/predict_iris",
    json = sample,
    timeout = 10
)

print(reponse.json())


{'prediction': 0, 'probabilities': [1.0, 0.0, 0.0]}


In [1]:
import torch

from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code = True)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code = True,
    dtype = (torch.float16 if torch.cuda.is_available() else torch.float32),
    device_map = "auto"

)


llm_model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [2]:
def generate_answer(question, max_new_tokens = 120):
  message =[
      {
          "role" : "system",
          "content" : "You are a helpful assistant"
      },
      {
          "role" : "user",
          "content" : question
      }
  ]

  inputs = tokenizer.apply_chat_template(
      message,
      tokenize = True,
      add_generation_prompt = True,
      enable_thinking = False,
      return_dict = True,
      return_tensors = "pt"
  )

  inputs = inputs.to(llm_model.device)

  with torch.no_grad():
    tokens = llm_model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        do_sample = True,
        temperature = 0.7,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id
    )

  answer = tokenizer.decode(tokens[0], skip_special_tokens = True)

  return answer




In [5]:
from pydantic import BaseModel, Field
class ChatRequest(BaseModel):
  question : str = Field(min_length = 1,max_length = 1024)

@app.post("/chat")

def chat(chat_request : ChatRequest):
  answer = generate_answer(chat_request.question)

  return {"answer" : answer}

In [8]:
response = requests.post(
    URL+"/chat",
    json = {
        "question" : "Explain REST APIs in 2 sentenses ? "
    },
    timeout = 100
)

print(response.json())

{'answer': 'system\nYou are a helpful assistant\nuser\nExplain REST APIs in 2 sentenses ? \nassistant\n<think>\n\n</think>\n\nREST APIs (Representational State Transfer APIs) are a set of HTTP methods that allow clients to communicate with a server in a structured and standardized way, typically using JSON or XML. They provide a simple and flexible way to build web applications and services.'}


In [9]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(
    title="Student Management API",
    description="FastAPI demonstration for Postman",
    version="1.0"
)

# Fictional student database
students = {
    101: {
        "name": "Raghava",
        "age": 20,
        "grade": "A",
        "department": "CSE"
    },
    102: {
        "name": "Raj",
        "age": 21,
        "grade": "B",
        "department": "ECE"
    },
    103: {
        "name": "Ram",
        "age": 22,
        "grade": "C",
        "department": "EEE"
    }
}


class Student(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=16, le=100)
    grade: str
    department: str


@app.get("/")
def home():
    return {"message": "Student API is running"}


@app.get("/students")
def get_all_students():
    return students


@app.get("/students/{student_id}")
def get_student(student_id: int):

    if student_id not in students:
        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    return students[student_id]


@app.post("/students", status_code=201)
def create_student(student: Student):

    new_id = max(students.keys(), default=100) + 1

    students[new_id] = student.model_dump()

    return {
        "student_id": new_id,
        "student": students[new_id]
    }


@app.put("/students/{student_id}")
def update_student(student_id: int, student: Student):

    if student_id not in students:
        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    students[student_id] = student.model_dump()

    return {
        "message": "Student updated",
        "student": students[student_id]
    }


@app.delete("/students/{student_id}")
def delete_student(student_id: int):

    if student_id not in students:
        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    del students[student_id]

    return {"message": "Student deleted"}


print("FastAPI application created successfully.")

FastAPI application created successfully.


In [10]:
import threading
import time
import uvicorn
import requests

PORT = 8000

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=PORT,
        log_level="warning"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

response = requests.get(
    f"http://127.0.0.1:{PORT}/",
    timeout=10
)

print(response.json())

{'message': 'Student API is running'}


In [11]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O cloudflared.deb

!dpkg -i cloudflared.deb

zsh:1: command not found: wget
zsh:1: command not found: dpkg


In [12]:
import subprocess
import threading
import re
import time

# Start Cloudflare Quick Tunnel
tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_lines = []

def read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.append(line)

threading.Thread(
    target=read_tunnel_output,
    daemon=True
).start()

public_url = None

for _ in range(30):
    output = "\n".join(tunnel_lines)

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        public_url = match.group(0)
        break

    if tunnel_process.poll() is not None:
        break

    time.sleep(1)

if public_url:
    print("Your public API URL:")
    print(public_url)
else:
    print("Tunnel URL not found. Recent output:")
    print("\n".join(tunnel_lines[-15:]))

Your public API URL:
https://valium-import-vocal-throughout.trycloudflare.com


In [13]:
if public_url:
    response = requests.get(
        public_url + "/students/101",
        timeout=30
    )

    print("Status:", response.status_code)
    print(response.json())

Status: 200
{'name': 'Raghava', 'age': 20, 'grade': 'A', 'department': 'CSE'}


In [14]:
tunnel_process.terminate()

try:
    tunnel_process.wait(timeout=10)
except subprocess.TimeoutExpired:
    tunnel_process.kill()

print("Public tunnel stopped.")

Public tunnel stopped.


In [16]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

from pydantic import BaseModel, Field

app = FastAPI(
    title="AI-Powered University API",
    description=(
        "Student management, machine learning, "
        "chatbot and RAG APIs"
    ),
    version="1.0.0"
)


# -----------------------------------------
# DATA MODELS
# -----------------------------------------

class StudentCreate(BaseModel):

    name: str = Field(min_length=2)
    age: int = Field(ge=16, le=100)
    department: str
    cgpa: float = Field(ge=0, le=10)


class StudentPatch(BaseModel):

    name: str | None = Field(default=None, min_length=2)
    age: int | None = Field(default=None, ge=16, le=100)
    department: str | None = None
    cgpa: float | None = Field(default=None, ge=0, le=10)


class IrisInput(BaseModel):

    sepal_length: float = Field(gt=0)
    sepal_width: float = Field(gt=0)
    petal_length: float = Field(gt=0)
    petal_width: float = Field(gt=0)


class ChatRequest(BaseModel):

    question: str = Field(min_length=1, max_length=1000)


class RAGRequest(BaseModel):

    question: str = Field(min_length=1, max_length=1000)


# -----------------------------------------
# IN-MEMORY DATABASE
# -----------------------------------------

students_db = {}

next_student_id = 101


# -----------------------------------------
# BASIC ENDPOINTS
# -----------------------------------------

@app.get("/")
def home():

    return {
        "message": "AI University API",
        "version": "1.0.0"
    }


@app.get("/health")
def health():

    return {
        "status": "ok"
    }


# -----------------------------------------
# STUDENT MANAGEMENT
# -----------------------------------------

@app.get("/students")
def list_students(department: str | None = None):

    results = []

    for student_id, student in students_db.items():

        if department is not None:

            if student["department"] != department:
                continue

        results.append({
            "student_id": student_id,
            **student
        })

    return results


@app.get("/students/{student_id}")
def get_student(student_id: int):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.post("/students", status_code=201)
def create_student(student: StudentCreate):

    global next_student_id

    student_id = next_student_id

    next_student_id += 1

    students_db[student_id] = student.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.put("/students/{student_id}")
def replace_student(
    student_id: int,
    student: StudentCreate
):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    students_db[student_id] = student.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.patch("/students/{student_id}")
def patch_student(
    student_id: int,
    update: StudentPatch
):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    changes = update.model_dump(exclude_unset=True)

    candidate = {
        **students_db[student_id],
        **changes
    }

    validated = StudentCreate.model_validate(candidate)

    students_db[student_id] = validated.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.delete("/students/{student_id}")
def delete_student(student_id: int):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    del students_db[student_id]

    return {
        "message": "Student deleted"
    }


# -----------------------------------------
# MACHINE LEARNING API
# -----------------------------------------

@app.post("/predict-iris")
def predict_iris(data: IrisInput):

    features = [[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]]

    prediction = classifier.predict(features)[0]

    probabilities = classifier.predict_proba(features)[0]

    return {
        "predicted_class": iris.target_names[prediction],
        "probabilities": {
            name: float(prob)
            for name, prob in zip(
                iris.target_names,
                probabilities
            )
        }
    }


# -----------------------------------------
# CHATBOT API
# -----------------------------------------

@app.post("/chat")
def chat(request: ChatRequest):

    answer = generate_answer(
        request.question
    )

    return {
        "question": request.question,
        "answer": answer
    }


# -----------------------------------------
# RAG API
# -----------------------------------------

@app.post("/rag")
def rag(request: RAGRequest):

    context = retrieve_context(
        request.question
    )

    prompt = f"""
Use only the following context.

Context:
{chr(10).join(context)}

Question:
{request.question}

If the answer is unavailable, say so.
"""

    answer = generate_answer(prompt)

    return {
        "question": request.question,
        "answer": answer,
        "context": context
    }


# -----------------------------------------
# TEST CLIENT
# -----------------------------------------

client = TestClient(app)

print("Application created successfully!")

print("Available endpoints:")

for route in app.routes:

    if hasattr(route, "methods"):

        print(
            route.path,
            route.methods
        )

Application created successfully!
Available endpoints:
/openapi.json {'HEAD', 'GET'}
/docs {'HEAD', 'GET'}
/docs/oauth2-redirect {'HEAD', 'GET'}
/redoc {'HEAD', 'GET'}
/ {'GET'}
/health {'GET'}
/students {'GET'}
/students/{student_id} {'GET'}
/students {'POST'}
/students/{student_id} {'PUT'}
/students/{student_id} {'PATCH'}
/students/{student_id} {'DELETE'}
/predict-iris {'POST'}
/chat {'POST'}
/rag {'POST'}


/Users/mukundanramesh/projects/ai_ml_course_bootcamp_practice/aiml_venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [17]:
import threading
import time
import uvicorn
import requests

PORT = 8000

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=PORT,
        log_level="warning"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

response = requests.get(
    f"http://127.0.0.1:{PORT}/",
    timeout=10
)

print(response.json())

ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


{'message': 'Student API is running'}


In [18]:
import subprocess
import threading
import re
import time

# Start Cloudflare Quick Tunnel
tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_lines = []

def read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.append(line)

threading.Thread(
    target=read_tunnel_output,
    daemon=True
).start()

public_url = None

for _ in range(30):
    output = "\n".join(tunnel_lines)

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        public_url = match.group(0)
        break

    if tunnel_process.poll() is not None:
        break

    time.sleep(1)

if public_url:
    print("Your public API URL:")
    print(public_url)
else:
    print("Tunnel URL not found. Recent output:")
    print("\n".join(tunnel_lines[-15:]))

FileNotFoundError: [Errno 2] No such file or directory: 'cloudflared'